# BirdCLEF+ 2026: Data exploration

Purpose of this notebook:

1. Inventory what the dataset actually contains.
2. Settle the key question: do the soundscapes carry real coordinates, or only site codes? This decides how precise the biodiversity map can be.
3. Get a first look at class balance and the multi-label structure, since both shape the modelling and evaluation choices later.

In [ ]:
import re
import pandas as pd

# Load datasets
train = pd.read_csv('../data/train.csv')
ss = pd.read_csv('../data/train_soundscapes_labels.csv')

print("train.csv rows:", len(train))
print("soundscape label rows:", len(ss))

train.csv rows: 35549
soundscape label rows: 1478


## The two kinds of audio

The dataset holds two very different sources, and the distinction shapes the whole project.

- **`train.csv`** describes the focal clips from Xeno-canto and iNaturalist. These are clean, usually a single animal, deliberately recorded.
- **`train_soundscapes_labels.csv`** describes labelled field soundscapes. These are messy, with several species calling at once and heavy background noise.

The gap between these two is the domain shift the model has to survive, since it trains mostly on the clean audio but is used on the messy audio.

In [ ]:
# Explore dataset columns
print("train columns:")
print(list(train.columns))
print()
print("soundscape columns:")
print(list(ss.columns))
print()
print(ss.head())

train columns:
['primary_label', 'secondary_labels', 'type', 'latitude', 'longitude', 'scientific_name', 'common_name', 'class_name', 'inat_taxon_id', 'author', 'license', 'rating', 'url', 'filename', 'collection']

soundscape columns:
['filename', 'start', 'end', 'primary_label']

                                    filename     start       end  \
0  BC2026_Train_0039_S22_20211231_201500.ogg  00:00:00  00:00:05   
1  BC2026_Train_0039_S22_20211231_201500.ogg  00:00:05  00:00:10   
2  BC2026_Train_0039_S22_20211231_201500.ogg  00:00:10  00:00:15   
3  BC2026_Train_0039_S22_20211231_201500.ogg  00:00:15  00:00:20   
4  BC2026_Train_0039_S22_20211231_201500.ogg  00:00:20  00:00:25   

                    primary_label  
0  22961;23158;24321;517063;65380  
1  22961;23158;24321;517063;65380  
2  22961;23158;24321;517063;65380  
3  22961;23158;24321;517063;65380  
4  22961;23158;24321;517063;65380  


## Where is the location information?

`train.csv` has `latitude` and `longitude`, so the focal clips carry real coordinates.

The soundscape table does not. It only has a filename, a start and end time, and the labels. The location is instead encoded inside the filename:

    BC2026_Train_0039_S22_20211231_201500.ogg
                      |    |        |
                      site date     time

So the map will be built at **site level**, using the site codes parsed out of the filenames. The `recording_location.txt` file gives the coordinate range of the recorder deployment sites, which anchors those sites to real positions.

In [ ]:
# Explore soundscape filenames
files = ss['filename'].unique()

sites = set()
dates = set()
hours = []

# Extract site, date, and hour information from filenames
for f in files:
    m = re.search(r'_(S\d+)_(\d{8})_(\d{6})', f)
    if m:
        sites.add(m.group(1))
        dates.add(m.group(2))
        hours.append(int(m.group(3)[:2]))

# Print summary statistics
print("distinct soundscape files:", len(files))
print("example filename:", files[0])
print()
print("distinct sites:", len(sites))
print(sorted(sites))
print()
print("distinct dates:", len(dates))
print("hours of day present:", sorted(set(hours)))

distinct soundscape files: 66
example filename: BC2026_Train_0039_S22_20211231_201500.ogg

distinct sites: 9
['S03', 'S08', 'S09', 'S13', 'S15', 'S18', 'S19', 'S22', 'S23']

distinct dates: 51
hours of day present: [0, 1, 2, 3, 4, 6, 7, 18, 19, 20, 21, 22, 23]


## Class balance

234 species across birds, amphibians, reptiles, mammals and insects. The concern is the long tail: species with very few recordings are the ones the model will miss, and they are also the ones that matter most for a diversity estimate.

In [ ]:
# Explore species counts
counts = train['primary_label'].value_counts()

print("species present in train.csv:", len(counts))
print("median clips per species:", int(counts.median()))
print("species with fewer than 10 clips:", int((counts < 10).sum()))
print("species with fewer than 5 clips:", int((counts < 5).sum()))
print()
print(train['class_name'].value_counts())

species present in train.csv: 206
median clips per species: 125
species with fewer than 10 clips: 25
species with fewer than 5 clips: 14

class_name
Aves        34799
Amphibia      451
Insecta       199
Mammalia       99
Reptilia        1
Name: count, dtype: int64


## Multi-label structure

A single five second segment can contain several species at once, with the labels separated by semicolons. This is why the task is multi-label rather than a single-class prediction, and why the evaluation uses ROC-AUC rather than plain accuracy.

In [ ]:
# Explore multi-label segments
label_counts = ss['primary_label'].astype(str).str.split(';').apply(len)

print("segments with more than one species:", int((label_counts > 1).sum()), "of", len(ss))
print("max species in a single segment:", int(label_counts.max()))
print()
print(ss['primary_label'].head())

segments with more than one species: 1322 of 1478
max species in a single segment: 10

0    22961;23158;24321;517063;65380
1    22961;23158;24321;517063;65380
2    22961;23158;24321;517063;65380
3    22961;23158;24321;517063;65380
4    22961;23158;24321;517063;65380
Name: primary_label, dtype: str


## Are the labels per segment or per file?

The soundscape table has one row per five second segment, but the first few rows all showed the same species. That could mean labels are applied to the whole file rather than to each segment individually, so every chunk gets tagged with everything present anywhere in the recording.

This matters for training. If the labels are file-level, then many segments are tagged with species that are not actually audible in them, which is a noisy training signal to plan around.

In [ ]:
# Explore label consistency within files
per_file = ss.groupby('filename')['primary_label'].nunique()

print("files where all segments share one label set:", int((per_file == 1).sum()))
print("files with varying labels across segments:", int((per_file > 1).sum()))
print()
print("segments per file (median):", int(ss.groupby('filename').size().median()))

files where all segments share one label set: 10
files with varying labels across segments: 56

segments per file (median): 24


## Findings

The metadata review was completed on **10 July 2026** using the BirdCLEF+ 2026 dataset.

### Location

The focal training clips include geographic coordinates. The `train.csv` file contains both `latitude` and `longitude` columns for all **35,549 recordings**.

The soundscape data does not include coordinates directly. Instead, the soundscape table contains the filename, start time, end time and species labels. The recording location is represented by a site code within each filename.

There are nine soundscape sites in total:

* S03
* S08
* S09
* S13
* S15
* S18
* S19
* S22
* S23

The map will therefore be created at site level. The sites will be positioned within the recorder deployment area described in `recording_location.txt`. This area covers part of the Pantanal in Mato Grosso do Sul, between approximately **-16.5 and -21.6 latitude** and **-55.9 and -57.6 longitude**.

Because the analysis includes only nine sites, the spatial sample is relatively small. It is suitable for comparing diversity between individual locations, but not for producing a continuous biodiversity surface or carrying out spatial statistics that require a much larger number of points.

The map should therefore be presented as a comparison of nine distinct recording sites rather than as a continuous spatial representation of biodiversity.

### Time

Recordings are available for the following hours of the day:

`0, 1, 2, 3, 4, 6, 7, 18, 19, 20, 21, 22 and 23`

The dataset also covers **51 separate recording dates**.

There are no recordings between approximately **08:00 and 17:00**. Most of the audio was collected during the night, dawn and dusk, which are periods when many of the species in the dataset are most vocally active.

This makes it possible to compare biodiversity across dawn, dusk and night. Given the limited number of recording sites, the temporal part of the analysis is likely to be more informative than the spatial component.

### Species and Class Balance

The `train.csv` file contains **206 species**, rather than the 234 species mentioned in the competition overview. This suggests that the full competition label set is not completely represented in the available training data.

The median number of clips per species is **125**. However:

* 25 species have fewer than 10 clips.
* 14 species have fewer than five clips.

The taxonomic distribution of the focal clips is shown below:

| Taxonomic Group | Number of Clips |
| --------------- | --------------: |
| Aves            |          34,799 |
| Amphibia        |             451 |
| Insecta         |             199 |
| Mammalia        |              99 |
| Reptilia        |               1 |

The dataset is heavily dominated by birds. Reptiles are represented by only one recording, which is not enough to support meaningful model training or evaluation for that group.

In practical terms, the project is mainly a bird classification task, with a much smaller number of non-avian examples.

Rather than overlooking this imbalance, it should be reported as an important finding. It also limits how confidently the final diversity measures can be interpreted as representing biodiversity beyond birds.

### Labelled Field Audio

The dataset contains **66 labelled soundscape files**, divided into **1,478 labelled five-second segments**. This is equal to approximately two hours of labelled field audio.

Of the 1,478 segments:

* 1,322 contain more than one species.
* Some segments contain as many as 10 species at the same time.
* Labels change between segments in 56 of the 66 soundscape files.

This shows that the annotations were generally applied at segment level rather than assigning the same labels to an entire recording.

The results confirm that this is a **multi-label classification problem**. This supports the use of evaluation measures such as ROC-AUC rather than relying on standard accuracy alone.

However, the amount of labelled field audio is limited. With only around two hours available, any conclusions about how well the model performs in real field conditions will need to be presented cautiously.

### Implications for the Project Plan

1. **Project scope**

   The project will be described as a bird-dominated acoustic monitoring study. Non-avian species will still be considered where the available data allows, but the imbalance between taxonomic groups will be clearly acknowledged.

2. **Temporal analysis**

   Greater emphasis will be placed on the temporal analysis. The dataset provides useful coverage across night, dawn and dusk, while the number of recording sites is limited.

3. **Spatial analysis**

   The spatial analysis will compare biodiversity across nine individual sites rather than attempting to create a continuous spatial surface.

4. **Field evaluation**

   Evaluation using the labelled soundscapes will be treated as a limited test of field performance. The available labelled field audio totals only around two hours, so any conclusions drawn from this evaluation will be appropriately qualified.
